# **Task 0.2.2 Transfer Learning from MNIST**

• Prepare a CNN of your choice and train it on the MNIST data. Report the accuracy<br>
• Use the above model as a pre-trained CNN for the SVHN dataset. Report the accuracy<br>
• In the third step you are performing transfer learning from MNIST to SVHN (optional).<br>

## **Prepare a CNN of your choice and train it on the MNIST data.**

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

#  Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
LR = 1e-4
MNIST_EPOCHS = 10
MODEL_PATH = "mnist_pretrained.pth"

# Data
# Resized MNIST to 32x32 and convert to 3 channels to match SVHN architecture
transform_mnist = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))])

transform_svhn = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Load MNIST
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform_mnist)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)
mnist_train_loader = DataLoader(mnist_train, batch_size=BATCH_SIZE, shuffle=True)
mnist_test_loader = DataLoader(mnist_test, batch_size=BATCH_SIZE, shuffle=False)

# --- Model ---
# Simple CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.LeakyReLU(0.01),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Train on MNIST
print(f"----Training on MNIST----")
model = SimpleCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(MNIST_EPOCHS):
    model.train()
    for images, labels in mnist_train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

    # Check Accuracy
    model.eval()
    correct, total = 0, 0
    test_loss = 0.0
    with torch.no_grad():
        for images, labels in mnist_test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, pred = outputs.max(1)
            total += labels.size(0)
            correct += pred.eq(labels).sum().item()
    avg_test_loss = test_loss / len(mnist_test_loader)
    test_acc = 100. * correct / total
    print(f"Epoch {epoch+1} | Test Loss: {avg_test_loss:.4f} | Test Acc: {test_acc:.2f}%")

# Save the trained weights
torch.save(model.state_dict(), MODEL_PATH)
print(f"Weights saved to {MODEL_PATH}\n")



----Training on MNIST----
Epoch 1 | Test Loss: 0.1051 | Test Acc: 96.66%
Epoch 2 | Test Loss: 0.0591 | Test Acc: 98.12%
Epoch 3 | Test Loss: 0.0418 | Test Acc: 98.65%
Epoch 4 | Test Loss: 0.0384 | Test Acc: 98.75%
Epoch 5 | Test Loss: 0.0390 | Test Acc: 98.65%
Epoch 6 | Test Loss: 0.0276 | Test Acc: 99.12%
Epoch 7 | Test Loss: 0.0282 | Test Acc: 99.15%
Epoch 8 | Test Loss: 0.0299 | Test Acc: 99.00%
Epoch 9 | Test Loss: 0.0209 | Test Acc: 99.31%
Epoch 10 | Test Loss: 0.0212 | Test Acc: 99.23%
Weights saved to mnist_pretrained.pth



## **Use the above model as a pre-trained CNN for the SVHN dataset.**

In [6]:
# Load SVHN
svhn_test = datasets.SVHN(root='./data', split='test', download=True, transform=transform_svhn)
svhn_test_loader = DataLoader(svhn_test, batch_size=BATCH_SIZE, shuffle=False)

print("--- Evaluating Pre-trained Model on SVHN ---")
# Create a model instance and load the weights
pretrained_model = SimpleCNN().to(DEVICE)
pretrained_model.load_state_dict(torch.load(MODEL_PATH))
pretrained_model.eval()

svhn_correct, svhn_total = 0, 0
with torch.no_grad():
    for images, labels in svhn_test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = pretrained_model(images)
        _, pred = outputs.max(1)
        svhn_total += labels.size(0)
        svhn_correct += pred.eq(labels).sum().item()

print(f"Final Accuracy on SVHN Dataset: {100. * svhn_correct / svhn_total:.2f}%")

--- Evaluating Pre-trained Model on SVHN ---
Final Accuracy on SVHN Dataset: 31.75%


## **Performing transfer learning from MNIST to SVHN**

In [8]:
# Load SVHN Training Data
svhn_train = datasets.SVHN(root='./data', split='train', download=True, transform=transform_svhn)
svhn_train_loader = DataLoader(svhn_train, batch_size=BATCH_SIZE, shuffle=True)

# Load the Pre-trained Model
tl_model = SimpleCNN().to(DEVICE)
tl_model.load_state_dict(torch.load("mnist_pretrained.pth"))

# Set up Fine-Tuning
optimizer = optim.Adam(tl_model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

# Fine-Tuning Loop
print("Fine-Tuning on SVHN")
tl_model.train()

for epoch in range(10):
    running_loss = 0.0
    for images, labels in svhn_train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = tl_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1} Fine-Tuning Loss: {running_loss/len(svhn_train_loader):.4f}")

#  Final Evaluation
tl_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in svhn_test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = tl_model(images)
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()

print(f"\nFinal Accuracy after Transfer Learning: {100.*correct/total:.2f}%")

Fine-Tuning on SVHN
Epoch 1 Fine-Tuning Loss: 1.5340
Epoch 2 Fine-Tuning Loss: 1.1124
Epoch 3 Fine-Tuning Loss: 0.9290
Epoch 4 Fine-Tuning Loss: 0.8227
Epoch 5 Fine-Tuning Loss: 0.7531
Epoch 6 Fine-Tuning Loss: 0.7040
Epoch 7 Fine-Tuning Loss: 0.6678
Epoch 8 Fine-Tuning Loss: 0.6386
Epoch 9 Fine-Tuning Loss: 0.6149
Epoch 10 Fine-Tuning Loss: 0.5940

Final Accuracy after Transfer Learning: 83.57%
